In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from kg_audit.entity.config_entity import (
    KGAuditPipelineConfig,
    TruthEvaluatorConfig,
    ModelTrainerConfig
)
from kg_audit.entity.artifact_entity import GraphRetrieverArtifact
from kg_audit.components.model_trainer import ModelTrainer
from kg_audit.components.truth_evaluator import TruthEvaluator

pipeline_config = KGAuditPipelineConfig()

# 1. Train & Calibrate the Belief Model (Produces final_model/model.pkl)
trainer_config = ModelTrainerConfig(pipeline_config)
trainer = ModelTrainer(trainer_config)
trainer_artifact = trainer.initiate_model_trainer()
print("=== MODEL TRAINER LEADERBOARD WINNER ===")
print(f"Best Model: {trainer_artifact.best_model_name}")
print(f"ROC-AUC: {trainer_artifact.roc_auc_score:.4f} | F1: {trainer_artifact.f1_score:.4f}")
print(f"Brier Score (Calibration Loss): {trainer_artifact.brier_score:.4f} | ECE: {trainer_artifact.expected_calibration_error:.4f}")
print(f"Promoted to Production: {trainer_artifact.is_model_accepted}\n")

# 2. Ground Truth Subgraph (Reversed node order to prove Canonical Alignment works!)
gt_nodes = [
    {"id": 10, "name": "Database"},
    {"id": 20, "name": "Server"},
    {"id": 30, "name": "User"}
]
gt_edges = [
    {"source": 30, "target": 20, "type": "USES", "weight": 1.0},
    {"source": 20, "target": 10, "type": "ACCESSES", "weight": 1.0}
]

retriever_artifact = GraphRetrieverArtifact(
    query_entities=["User", "Database"],
    extracted_nodes=gt_nodes,
    extracted_edges=gt_edges,
    subgraph_file_path="mock_path.json",
    retrieval_status=True
)

evaluator = TruthEvaluator(TruthEvaluatorConfig(pipeline_config))

# 3. Test Case A: Hallucinated Response (Invented shortcut User -> Database)
llm_nodes_hallucinated = [
    {"id": 0, "name": "User"},
    {"id": 1, "name": "Server"},
    {"id": 2, "name": "Database"}
]
llm_edges_hallucinated = [
    {"source": 0, "target": 1, "type": "USES", "weight": 1.0},
    {"source": 1, "target": 2, "type": "ACCESSES", "weight": 1.0},
    {"source": 0, "target": 2, "type": "DIRECT_ACCESS", "weight": 1.0}
]

hallucinated_res = evaluator.evaluate_truth(retriever_artifact, llm_nodes_hallucinated, llm_edges_hallucinated)
print("=== TEST CASE A: HALLUCINATED CLAIM ===")
print(f"8D Feature Vector X: {[round(x, 4) for x in hallucinated_res.feature_vector]}")
print(f"Calibrated Belief Score: {hallucinated_res.final_accuracy_score:.4f} | Verdict: {hallucinated_res.decision_status}\n")

# 4. Test Case B: Factual Response (Matches Ground Truth)
llm_edges_factual = [
    {"source": 0, "target": 1, "type": "USES", "weight": 1.0},
    {"source": 1, "target": 2, "type": "ACCESSES", "weight": 1.0}
]

factual_res = evaluator.evaluate_truth(retriever_artifact, llm_nodes_hallucinated, llm_edges_factual)
print("=== TEST CASE B: FACTUAL CLAIM ===")
print(f"8D Feature Vector X: {[round(x, 4) for x in factual_res.feature_vector]}")
print(f"Calibrated Belief Score: {factual_res.final_accuracy_score:.4f} | Verdict: {factual_res.decision_status}")

e:\00 My Work\EpistemeGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== MODEL TRAINER LEADERBOARD WINNER ===
Best Model: RandomForest
ROC-AUC: 0.9690 | F1: 0.9496
Brier Score (Calibration Loss): 0.0520 | ECE: 0.0279
Promoted to Production: True



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1789.22it/s]


=== TEST CASE A: HALLUCINATED CLAIM ===
8D Feature Vector X: [2.7954, 2.0, 0.8382, 0.8421, 0.9277, 0.85, 0.3333, 0.9]
Calibrated Belief Score: 0.0873 | Verdict: HALLUCINATED

=== TEST CASE B: FACTUAL CLAIM ===
8D Feature Vector X: [0.0, 0.0, 0.0, 0.8421, 0.9277, 0.85, 0.0, 0.9]
Calibrated Belief Score: 0.8252 | Verdict: VERIFIED
